In [ ]:
import numpy as np
from scipy.sparse import csr_matrix
from scipy.linalg import qr
import time
from typing import Optional, Dict, Any

# Number of observation
N = 40000

# Dimension of input
X1 = 7
# Dimension of hidden layer 1
H1 = 80
# Dimension of hidden layer 2
H2 = 40
# Dimension of output 4
H3 = 1

# Input X (unseeded randomness)
X = np.random.uniform(-1, 1, size=(X1, N))

# Number of parameters
M =(X1+1)*H1+(H1+1)*H2+(H2+1)*H3



# Output Y
Y = 10*np.sin(np.pi*X[0,:]*X[1,:]) + 20*(X[2,:]-0.5)**2+10*X[3,:]+5*X[4,:] # simple target

# Noisy Data
Data = Y + 0.05*np.std(Y)*np.random.normal(0,1,size=N)

N1 = 1000
XY = np.random.uniform(-1, 1, size=(X1, N1))
YY  = 10*np.sin(np.pi*XY[0,:]*XY[1,:]) + 20*(XY[2,:]-0.5)**2+10*XY[3,:]+5*XY[4,:]


In [ ]:
# Activation function
def tanh(z):
    return np.tanh(z)

# Derivative of activation function: tanh'(z) = 1 - a^2
def tanh_prime(a):
    return 1.0 - a**2


# -------------------------
# Neural Network as Function approximation
# -------------------------
def NN(x, param):
    # x: (X1, 1)

    # Layer 1
    W1 = param[0:X1 * H1].reshape((H1, X1))
    b1 = param[X1 * H1:(X1 + 1) * H1].reshape((H1, 1))

    # Layer 2
    W2_start = (X1 + 1) * H1
    W2_end = W2_start + H1 * H2

    W2 = param[W2_start:W2_end].reshape((H2, H1))
    b2 = param[W2_end:W2_end + H2].reshape((H2, 1))

    # Output layer
    W3_start = W2_end + H2
    W3_end = W3_start + H2 * H3

    W3 = param[W3_start:W3_end].reshape((H3, H2))
    b3 = param[W3_end:W3_end + H3].reshape((H3, 1))

    # Forward pass
    z1 = W1 @ x + b1
    a1 = tanh(z1)

    z2 = W2 @ a1 + b2
    a2 = tanh(z2)

    y = W3 @ a2 + b3

    cache = {
        "x": x,
        "a1": a1,
        "a2": a2,
        "z1": z1,
        "z2": z2,
        "y": y
    }

    return y, cache


# -------------------------
# Column indexing: X1 -> H1 -> H2 -> OUT
# -------------------------
def index_W1(j, i):
    return j * X1 + i

def index_b1(j):
    return X1 * H1 + j

def index_W2(j, i):
    return (X1 + 1) * H1 + j * H1 + i

def index_b2(j):
    return (X1 + 1) * H1 + H1 * H2 + j

def index_W3(i):
    return (X1 + 1) * H1 + (H1 + 1) * H2 + i

def index_b3():
    return (X1 + 1) * H1 + (H1 + 1) * H2 + H2

# -------------------------
# Compute the Jacobian J and the residual vector e = d - o
# -------------------------
def J_R(param):
    """
    param: Trainable parameters
    Returns: Jacobian & Residuals
    """

    # Extract W2
    W2_start = (X1 + 1) * H1
    W2_end = W2_start + H1 * H2
    W2 = param[W2_start:W2_end].reshape((H2, H1))

    # Extract output weights W3
    W3_start = W2_end + H2
    W3_end = W3_start + H2 * H3
    W3 = param[W3_start:W3_end].reshape((H3, H2))

    N = X.shape[1]

    J = np.zeros((N, M), dtype=float)
    e = np.zeros((N,), dtype=float)

    for p in range(N):
        x = X[:, [p]]      # (X1, 1)
        d = Data[p]        # scalar

        y, cache = NN(x, param)

        # Residual
        e[p] = d - y[0, 0]

        a1 = cache["a1"]   # (H1, 1)
        a2 = cache["a2"]   # (H2, 1)

        # ---------- Jacobian backprop deltas for residual e = d - y ----------
        # Since e = d - y => de/dy = -1
        delta3 = -1.0  # output layer scalar

        # Hidden layer 2 delta: (H2, 1)
        delta2 = (W3.T * delta3) * tanh_prime(a2)

        # Hidden layer 1 delta: (H1, 1)
        delta1 = (W2.T @ delta2) * tanh_prime(a1)

        # ---------- Fill Jacobian row ----------
        # Layer 1 weights W1: input is x
        for j in range(H1):
            for i in range(X1):
                J[p, index_W1(j, i)] = -float(delta1[j, 0]) * float(x[i, 0])

        # Layer 1 biases b1
        for j in range(H1):
            J[p, index_b1(j)] = -float(delta1[j, 0])

        # Layer 2 weights W2: input is a1
        for j in range(H2):
            for i in range(H1):
                J[p, index_W2(j, i)] = -float(delta2[j, 0]) * float(a1[i, 0])

        # Layer 2 biases b2
        for j in range(H2):
            J[p, index_b2(j)] = -float(delta2[j, 0])

        # Output weights W3: input is a2
        for i in range(H2):
            J[p, index_W3(i)] = -float(delta3) * float(a2[i, 0])

        # Output bias b3
        J[p, index_b3()] = -float(delta3)

    return J, e

# Calculate Mean Square Error
def MSE(XX,YY,param):
    DD , dummy = NN(XX,param)
    return np.mean((DD - YY) ** 2)

# Deterministic subspace adequacy monitor
def eta_sub(V, g) :
    """
    Evaluate projected-gradient energy ratio
    """
    norm = float(g @ g)
    if norm == 0.0:
        return 1.0      
    proj = V.T @ g
    return float(proj @ proj) / norm

def krylov(J, v):
    return J.T @ (J @ v)

# Stochastic Hessian-based random probes
def random_hessian_probes(J, k):
    """
    Generate Stochastic Hessian-based random probes
    """
    m, n = J.shape
    W = np.random.standard_normal((n, k))
    D = J.T @ (J @ W)  # (n×k)
    norms = np.linalg.norm(D, axis=0, keepdims=True)
    norms[norms == 0.0] = 1.0
    D = D / norms
    return D


# Expand subspace by updating QR
def qr_append_columns(Q, R, D, tol = 1e-12):
    """
    Incrementally update thin QR for Wnew=[W,D] given W=QR.
    """
    if D.ndim == 1:
        D = D[:, None]

    n, s = Q.shape
    sR, p = R.shape

    b = D.shape[1]
    R_new = np.hstack([R, np.zeros((s, b))])

    for j in range(b):
        d = D[:, j]

        a = Q.T @ d
        r = d - Q @ a

        for _ in range(2):
            a2 = Q.T @ r
            r -= Q @ a2
            a += a2

        nr = np.linalg.norm(r)
        if nr > tol:
            q_new = r / nr
            Q = np.column_stack([Q, q_new])

            R_new = np.vstack([R_new, np.zeros((1, R_new.shape[1]))])
            R_new[:-1, p + j] = a
            R_new[-1,  p + j] = nr
        else:
            R_new[:, p + j] = a

        s = Q.shape[1]

    return Q, R_new

class LanczosState:
    def __init__(self, hvp_fn, q0: np.ndarray, reorth: int = 1, tol: float = 1e-12):
        self.hvp = hvp_fn
        self.reorth = reorth
        self.tol = tol

        q0 = np.asarray(q0, float).reshape(-1)
        nq = np.linalg.norm(q0)
        if nq == 0.0:
            raise ValueError("q0 must be nonzero.")
        self.q_prev = np.zeros_like(q0)
        self.q = q0 / nq
        self.beta_prev = 0.0
        self.Qk = [self.q.copy()]

    def step(self) -> np.ndarray:
        q = self.q
        w = self.hvp(q)
        if self.beta_prev != 0.0:
            w -= self.beta_prev * self.q_prev

        alpha = float(q @ w)
        w -= alpha * q

        if self.reorth > 0:
            Qmat = np.column_stack(self.Qk)
            for _ in range(self.reorth):
                c = Qmat.T @ w
                w -= Qmat @ c

        beta = np.linalg.norm(w)
        if beta <= self.tol:
            return np.zeros_like(q)

        q_next = w / beta
        self.q_prev = q
        self.q = q_next
        self.beta_prev = beta
        self.Qk.append(q_next.copy())
        return q_next


def reduced_subspace_lm_one_iteration(
    J: np.ndarray,
    r: np.ndarray,
    prev: np.ndarray,
    lambda0: float = 10.0,
    delta: float = 1e-8,
    eta_min: float = 0.99,
    max_total_vecs: int = int(M*0.1),
    qr_reorth: int = 1,
    qr_tol: float = 1e-12,
    lanczos_reorth: int = 1,
    lanczos_tol: float = 1e-5,
):
    """
    NO subspace compression (V = Q always).

    Initial basis:
      [recent step, 1% random Hessian probes]

    Expansion basis (updated as requested):
      +2% Krylov vectors + 1% random Hessian probes
      (truncated if max_total_vecs would be exceeded)
    """
    J = np.asarray(J, float)
    r = np.asarray(r, float).reshape(-1)
    m, n = J.shape
    lam = float(lambda0)

    g = J.T @ r
    ng = np.linalg.norm(g)
    gdir = g / ng

    hvp = lambda v: krylov(J, v)
    lanczos = LanczosState(hvp, q0=gdir, reorth=lanczos_reorth, tol=lanczos_tol)
   
    # initial: recent step & 1% dimension of randomize probs
    blocks = []
    expansions = 0
    
    if prev is not None:
        prev = np.asarray(prev, dtype=float)
    
        if prev.size > 0:
            # Convert (M,) -> (M,1)
            if prev.ndim == 1:
                prev = prev[:, None]
    
            if prev.ndim != 2:
                raise ValueError(
                    f"prev must be a vector or matrix, got shape {prev.shape}"
                )
    
            if prev.shape[0] != n:
                raise ValueError(
                    f"prev has {prev.shape[0]} rows, but J has {n} parameters"
                )
    
            blocks.append(prev)
            expansions = 1
    
    current_cols = sum(b.shape[1] for b in blocks)
    
    k_rand0 = min(
        int(n * 0.01)-1,
        max_total_vecs - current_cols
    )
    
    if k_rand0 > 0:
        blocks.append(random_hessian_probes(J, k_rand0))


    W0 = np.hstack(blocks) if blocks else np.zeros((n, 0))

    Q, R, _ = qr(W0, mode="economic", pivoting=True)
    V = Q
    eta = eta_sub(V, g)
    print(eta, V.shape[1])

    total_vecs = W0.shape[1]
    

    while eta < eta_min and total_vecs < max_total_vecs:
        add_cap = max_total_vecs - total_vecs
        new_cols = []

        # add up to 2% dimension Lanczos/Krylov vectors
        k_krylov = min(int(M*0.02), add_cap)
        for _ in range(k_krylov):
            q_next = lanczos.step()
            if np.linalg.norm(q_next) > 0:
                new_cols.append(q_next[:, None])

        used = sum(c.shape[1] for c in new_cols)
        add_cap2 = max_total_vecs - total_vecs - used

        # add up to 1% dimension random probes
        k_rand = min(int(M*0.01), add_cap2)
        if k_rand > 0:
            new_cols.append(random_hessian_probes(J, k_rand))

        if not new_cols:
            break

        D = np.hstack(new_cols)
        Q, R = qr_append_columns(Q, R, D, tol=qr_tol)

        total_vecs += D.shape[1]
        expansions += 1
        V = Q
        eta = eta_sub(V, g)
        print(eta, V.shape[1])

    Jr = J @ V
    U, svals, ZT = np.linalg.svd(Jr, full_matrices=False)
    Z = ZT.T

    sigma2 = svals**2
    d_diag = np.maximum(sigma2, delta)
    B_diag = sigma2 + lam * d_diag

    rhs = -(svals * (U.T @ r))
    y = rhs / (B_diag)
    s = V @ (Z @ y)
    return s,g
    
# ------------------------------------------------------------
# Symmetric tridiagonal solver
# ------------------------------------------------------------
def solve_tridiagonal(diag, off, rhs):
    diag = np.asarray(diag, dtype=float).copy()
    off = np.asarray(off, dtype=float)
    rhs = np.asarray(rhs, dtype=float).copy()

    n = diag.size
    if n == 1:
        return rhs / diag

    for i in range(1, n):
        m = off[i - 1] / diag[i - 1]
        diag[i] -= m * off[i - 1]
        rhs[i] -= m * rhs[i - 1]

    x = np.empty(n, dtype=float)
    x[-1] = rhs[-1] / diag[-1]
    for i in range(n - 2, -1, -1):
        x[i] = (rhs[i] - off[i] * x[i + 1]) / diag[i]
    return x


# ------------------------------------------------------------
# Build Krylov cache for scaled Marquardt system
#   (J^T J + lamda D) y = J^T e
# with D = diag(diag(J^T J))
# ------------------------------------------------------------
def build_marquardt_krylov_cache(
    J,
    g,
    d_diag,
    max_dim,

    lanczos_tol=1e-5,
):
    d_safe = np.maximum(np.asarray(d_diag, dtype=float), 1e-14)
    d_inv_sqrt = 1.0 / np.sqrt(d_safe)

    q0 = d_inv_sqrt * g
    q0_norm = np.linalg.norm(q0)
    n = J.shape[1]

    if q0_norm == 0.0:
        return {
            "Q": np.empty((n, 0)),
            "alpha": np.empty(0),
            "beta": np.empty(0),
            "q0_norm": 0.0,
            "d_inv_sqrt": d_inv_sqrt,
        }

    Q = np.empty((n, max_dim), dtype=float)
    alpha = np.empty(max_dim, dtype=float)
    beta = np.empty(max_dim - 1, dtype=float) if max_dim > 1 else np.empty(0, dtype=float)

    q = q0 / q0_norm
    q_prev = np.zeros_like(q)
    beta_prev = 0.0
    Q[:, 0] = q

    k = 1
    for j in range(max_dim):
        v = d_inv_sqrt * q
        w = J.T @ (J @ v)
        w = d_inv_sqrt * w

        if j > 0:
            w -= beta_prev * q_prev

        aj = float(q @ w)
        w -= aj * q
        alpha[j] = aj

        bj = np.linalg.norm(w)

        if j == max_dim - 1 or bj <= lanczos_tol:
            k = j + 1
            break

        beta[j] = bj
        q_prev = q
        q = w / bj
        beta_prev = bj
        Q[:, j + 1] = q
        k = j + 2
    print(k)
    return {
        "Q": Q[:, :k],
        "alpha": alpha[:k],
        "beta": beta[:k - 1],
        "q0_norm": q0_norm,
        "d_inv_sqrt": d_inv_sqrt,
    }


# ------------------------------------------------------------
# Solve projected step from cached Krylov basis
# ------------------------------------------------------------
def marquardt_step_from_krylov_cache(cache, lamda):
    Q = cache["Q"]
    alpha = cache["alpha"]
    beta = cache["beta"]
    q0_norm = cache["q0_norm"]
    d_inv_sqrt = cache["d_inv_sqrt"]

    if len(alpha) == 0:
        return np.zeros_like(d_inv_sqrt)

    rhs = np.zeros(len(alpha), dtype=float)
    rhs[0] = q0_norm

    diag = alpha + lamda
    z = solve_tridiagonal(diag, beta, rhs)

    step = d_inv_sqrt * (Q @ z)
    return step

for m in range(30):
    print(m)
    init = np.random.random(M)
   
    print('-------------------------------------------Classical LM---------------------------------------------------')
    lamda = 10
    # Initial guess params
    new = init
    y_old = MSE(X,Data,new)
    validate = MSE(XY,YY,new)
    sucess = True
    print(y_old,validate)
    avg1 = 0;
    diff = 0    
    for i in range(1000):
        old = new;
        if sucess:
            J, dy_new = J_R(new);
            d =  np.eye(J.shape[1])
    
        start_time = time.perf_counter()
        A = J.T@J + lamda*d
        b = J.T@dy_new   
        s = np.linalg.lstsq(A, b, rcond=None)[0]
        new = old + s
        y_new = MSE(X,Data,new)

        if y_old-y_new >= 0:
            diff = y_old-y_new
            y_old = y_new  
            sucess = True
            lamda = lamda/2
        else:
            sucess = False
            new = old
            lamda = lamda*5
        end_time = time.perf_counter()
        validate = MSE(XY,YY,new)
        print(i,y_old,validate,end_time - start_time)
        avg1 = avg1 + (end_time - start_time)
    
        if diff <5e-3 and sucess:
            break
    
    print(i+1,avg1/(i+1))
    print('-------------------------------------------------END------------------------------------------------------')

    print('-------------------------------------------Krylov LM---------------------------------------------------')
    lamda = 10
    # Initial guess params
    new = init
    y_old = MSE(X,Data,new)
    validate = MSE(XY,YY,new)
    sucess = True
    print(y_old,validate)
    avg1 = 0;
    diff = 0    
    for i in range(200):
        old = new
        if sucess:
            J, dy_new = J_R(new)
            d =  np.ones(J.shape[1])   # diagonal of J^T J

        start_time = time.perf_counter()
        if sucess:
            cache = build_marquardt_krylov_cache(
                    J=J,
                    g=J.T @ dy_new,
                    d_diag=d,
                    max_dim=int(M*0.1)
                )
        s = marquardt_step_from_krylov_cache(cache, lamda)
        new = old + s
        y_new = MSE(X,Data,new)

        if y_old - y_new >= 0:
            diff = y_old-y_new
            y_old = y_new
            sucess = True
            lamda = lamda /2

        else:
            sucess = False
            new = old
            lamda = lamda * 5
        end_time = time.perf_counter()
        validate = MSE(XY,YY,new)
        print(i, y_old,validate, end_time - start_time)
        avg1 = avg1 + (end_time - start_time)

        if diff <5e-3 and sucess:
            break

    print(i+1,avg1/(i+1))
    print('-------------------------------------------------END------------------------------------------------------')
   
    print('------------------------------------Reduced subspace LM---------------------------------------------------')
   
    lamda = 10
    # Initial guess params
    new = init
    y_old = MSE(X,Data,new)
    validate = MSE(XY,YY,new)
    sucess = True
    print(y_old,validate)
    avg1 = 0;
    diff = 0    
    s = None
    for i in range(1000):

        old = new
        if sucess:
            J, dy_new = J_R(new);
        start_time = time.perf_counter()
       
        s, grad = reduced_subspace_lm_one_iteration(J, dy_new,prev = s, lambda0=lamda)
        alpha = 1e-5
        t=1
        beta = 0.5
        descent = np.dot(s,grad)
        for j in range(10):
            new = old-t*s
            y_new = MSE(X,Data,new)
            if y_new-y_old <= alpha*t*descent:  
                diff = y_old-y_new
                y_old = y_new  
                sucess = True
                lamda = lamda/2
                break;
            else:
                sucess = False
                t = t*beta
                new = old
            
        if not sucess:
            lamda = lamda*5

        end_time = time.perf_counter()
        validate = MSE(XY,YY,new)
        print(i,y_old,validate,end_time - start_time)
        avg1 = avg1 + (end_time - start_time)
        if  diff<5e-3 and sucess:
            break

    print(i+1,avg1/(i+1))
    print('-------------------------------------------------END------------------------------------------------------') 